# Toxicité moléculaire avec GNN (version améliorée)

**Nom :** ............................................................  
**Prénom :** ........................................................  

Ce notebook montre un modèle GNN amélioré avec DeepChem.

In [1]:
import deepchem as dc
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (c:\Doc_local\introduction_IA\.venv\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (c:\

## Chargement dataset Tox21

In [2]:
# 1. On crée le convertisseur en graphes
featurizer = dc.feat.ConvMolFeaturizer()

# 2. On charge Tox21 en forçant l'utilisation de ce convertisseur
import os
local_cache = "./datasets/tox21_cache"
os.makedirs(local_cache, exist_ok=True)
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=featurizer, save_dir=local_cache, data_dir=local_cache)
train_dataset, valid_dataset, test_dataset = datasets


## Modèle GNN amélioré

In [3]:
model = dc.models.GraphConvModel(
    n_tasks=len(tasks),
    mode='classification',
    dropout=0.3,
    batch_size=64,
    learning_rate=5e-4
)

## Entraînement

In [4]:
model.fit(train_dataset, nb_epoch=50)

KeyboardInterrupt: 

## Évaluation

In [ ]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
print(model.evaluate(test_dataset, [metric]))

{'roc_auc_score': 0.6969994163309748}


## Test sur molécules

In [ ]:
smiles_a_tester = {
    "Paracétamol": "CC(=O)NC1=CC=C(O)C=C1",
    "Valdécoxib": "CC1=C(C(=NO1)C2=CC=CC=C2)C3=CC=C(C=C3)S(=O)(=O)N"
}

noms_molecules = list(smiles_a_tester.keys())
smiles = list(smiles_a_tester.values())

graphes_medocs = featurizer.featurize(smiles)
dataset = dc.data.NumpyDataset(X=graphes_medocs)

# Prédictions brutes
predictions = model.predict(dataset)

for i, nom in enumerate(noms_molecules):
    print(f"\n=== PROFIL DE TOXICITÉ : {nom} ===")
    
    # Pour chaque molécule, on regarde ses 12 scores
    for index, nom_cible in enumerate(tasks):
        # On regarde la probabilité d'être toxique (indice 1)
        probabilite = predictions[i][index][1] 
        
        if probabilite > 0.5:
            print(f"⚠️ DANGER sur {nom_cible:10s} : {probabilite*100:.1f}%")
        else:
            print(f"  Sûr    sur {nom_cible:10s} : {probabilite*100:.1f}%")



=== PROFIL DE TOXICITÉ : Paracétamol ===
  Sûr    sur NR-AR      : 27.2%
  Sûr    sur NR-AR-LBD  : 9.5%
⚠️ DANGER sur NR-AhR     : 77.0%
  Sûr    sur NR-Aromatase : 13.7%
⚠️ DANGER sur NR-ER      : 71.8%
⚠️ DANGER sur NR-ER-LBD  : 78.0%
⚠️ DANGER sur NR-PPAR-gamma : 63.6%
⚠️ DANGER sur SR-ARE     : 56.0%
⚠️ DANGER sur SR-ATAD5   : 72.5%
  Sûr    sur SR-HSE     : 36.2%
⚠️ DANGER sur SR-MMP     : 61.4%
  Sûr    sur SR-p53     : 46.5%

=== PROFIL DE TOXICITÉ : Valdécoxib ===
  Sûr    sur NR-AR      : 22.5%
  Sûr    sur NR-AR-LBD  : 38.9%
⚠️ DANGER sur NR-AhR     : 82.4%
⚠️ DANGER sur NR-Aromatase : 78.6%
  Sûr    sur NR-ER      : 37.4%
  Sûr    sur NR-ER-LBD  : 30.2%
⚠️ DANGER sur NR-PPAR-gamma : 71.5%
  Sûr    sur SR-ARE     : 44.6%
  Sûr    sur SR-ATAD5   : 27.5%
  Sûr    sur SR-HSE     : 20.0%
⚠️ DANGER sur SR-MMP     : 77.9%
  Sûr    sur SR-p53     : 25.5%


## Partie 4 : Bilan sur l'amélioration du modèle

### Questions :
1. **Hyperparamètres** : Dans ce modèle amélioré, nous avons ajusté le `dropout` à 0.3 et le `learning_rate` à 5e-4. À quoi sert la technique du *dropout* pendant l'entraînement d'un réseau de neurones ?
2. **Durée de l'entraînement** : Le paramètre `nb_epoch` est passé de 10 à 50. Quel est l'avantage de laisser le modèle s'entraîner plus longtemps, et quel est le risque principal si on l'entraîne trop (surapprentissage ou overfitting) ?
3. **Score et limites** : Le score ROC-AUC s'est amélioré (environ 0.697) par rapport au modèle de base. Pourquoi est-il si difficile d'obtenir un modèle qui prédit à 100% la toxicité d'une molécule sur un organisme vivant ?

*(Double-cliquez sur cette cellule pour rédiger vos réponses ci-dessous :)*

* **Réponse 1 :** ...
* **Réponse 2 :** ...
* **Réponse 3 :** ...
